In [ ]:
from sateline_ml import (
    initialize_ee,
    process_download,
    merge_all_bands,
    floating_debris_detection,
    create_mask_tif_from_shp,
)

tif='mydate/2022-04-01_2022-04-03_02-49-10.tif'
shp='mydate/shp/taihu.shp'

In [1]:
rough_thr = max(min(0.25,  0.6), -0.5)
print(rough_thr)

0.25


In [ ]:
import pandas as pd
import plotly.express as px 

df = pd.read_csv('spectral_library/plastic_spectra.csv', index_col=0)
df.stack().reset_index()

fig = px.line(df.stack().reset_index(), x='wavelength_nm', y=0, color='level_1')
fig.show()

In [ ]:
from sateline_ml.notebook_utils import organize_gee_outputs

result = organize_gee_outputs(
    root_dir="gee_nature3",
    target_dir="target_n",
    require_key="best",
    size_limit=1 * 1024 * 1024,
    dry_run=False,
)
print(f"renamed={len(result['renamed'])}, unchanged={len(result['unchanged'])}, moved={len(result['moved'])}")

In [ ]:
from sateline_ml.notebook_utils import filter_and_move_tifs
import re

selected, skipped = filter_and_move_tifs(
    source_dir="target_n",
    target_dir="tif",
    pattern=re.compile(r"_\d{4}-(07|08|09)_final.*\.tif$", re.IGNORECASE),
    size_limit=None,
    dry_run=False,
)
print(f"selected={len(selected)}, skipped={len(skipped)}")

In [ ]:

musk_tif=create_mask_tif_from_shp(shp,tif,'mydate/taihu_mask.tif')
floating_debris_detection(
    input_tif=tif,
    mask_tif=musk_tif,     # still required for pseudo-labels
    shp_file=shp,                   # no shapefile needed
    output_csv="mydate/run/detections.csv",
    preview_png="mydate/run/preview.png",
    debug_dir="mydate/run",   # 新增：步骤可视化目录
    log_file="mydate/run/run1.log",
    epochs=8,
    water_ndwi_thresh=0.03,          # tweak if water not picked up
    floating_thresh=0.55,
    suspended_thresh=0.55,
)



In [ ]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())


In [ ]:
import lightgbm as lgb

In [ ]:
import pandas as pd

df = pd.read_csv('spectral_library/plastic_spectra.csv')
df.head()

In [ ]:
import plotly.express as px

In [ ]:
fig=px.line(df.set_index('wavelength_nm').stack().reset_index(),
            x='wavelength_nm',color='level_1',y=0,template='simple_white',height=400,width=500)
fig.update_yaxes(title='absorb')
fig.show()

In [ ]:
from sateline_ml.extract_spectra_from_plot import extract_curves_from_image


plot_path='sp'
files = [f for f in os.listdir(plot_path) if f.endswith('.png') and not f.startswith('._')]

images = [ f"{plot_path}/{f}" for f in files ]
for img_path in images:
    print(img_path)
    df = extract_curves_from_image(
        img_path,
        x_min=1000, x_max=2500, y_min=0, y_max=1,
        crop_box_frac=None, k_colors=5,
        wavelength_step=2.0, min_pixels_per_curve=400
    )
    out_csv = img_path.rsplit(".",1)[0] + "_spectra.csv"
    df.to_csv(out_csv, index=False)


In [ ]:
from sateline_ml.notebook_utils import build_spectral_library_from_csvs

df_sp = build_spectral_library_from_csvs(
    csv_dir='sp',
    out_csv='spectral_library/plastic_spectra.csv',
)
df_sp

In [ ]:
# 已迁移为 scripts/bbox_from_lake_shp.py
# 用法示例：
#   python scripts/bbox_from_lake_shp.py /path/to/lakes.shp out.csv \
#       --out-shp-dir shplist/shp --group-field Lake_name --id-field Hylak_id

In [ ]:
# 已迁移为 scripts/bbox_from_lake_shp.py
# 用法示例：
#   python scripts/bbox_from_lake_shp.py /path/to/lakes.shp out.csv \
#       --out-shp-dir shplist/shp --group-field Lake_name --id-field Hylak_id

In [ ]:
# 已迁移为 scripts/xlsx_to_shapefiles.py
# 用法示例：
#   python scripts/xlsx_to_shapefiles.py target_lake_info.xlsx shplist/shp

In [ ]:
shp='HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10_shp/HydroLAKES_polys_v10.shp'
out_csv='hybas_lake_as_lev06_v1c.csv'
# group_field='name'

df=compute_bbox(gpd.read_file(shp), )
df

df2=gpd.read_file(shp)
df2.to_excel('HydroLAKES_polys_v10.csv')
# df.to_csv(out_csv, index=False, encoding="utf-8-sig")

In [ ]:

df3 = pd.merge(df, df2,on='Hylak_id')

df3

,feature_id,Hylak_id,min_lon,max_lon,min_lat,max_lat,Lake_name,Country,Continent,Poly_src,...,Vol_src,Depth_avg,Dis_avg,Res_time,Elevation,Slope_100,Wshd_area,Pour_long,Pour_lat,geometry
0,0,1,46.738614,54.052508,36.571189,47.130753,Caspian Sea,Russia,Europe,SWBD,...,1,200.5,8110.642,107883.0,-29,-1.00,1404108.0,47.717708,45.591934,"POLYGON ((49.96181 37.43847, 49.96457 37.44022..."
1,1,2,-125.142414,-117.456724,64.783399,67.038963,Great Bear,Canada,North America,CanVec,...,1,72.2,535.187,47577.7,145,-1.00,147665.4,-123.505546,65.138384,"POLYGON ((-119.78782 67.03574, -119.78637 67.0..."
2,2,3,-117.750874,-108.674534,60.808669,62.958619,Great Slave,Canada,North America,CanVec,...,1,59.1,4350.692,4203.2,148,-1.00,995312.3,-117.617115,61.311727,"POLYGON ((-109.93976 62.95851, -109.93831 62.9..."
3,3,4,-99.268558,-96.001631,50.208691,53.880490,Winnipeg,Canada,North America,CanVec,...,1,11.9,2244.727,1464.3,215,-1.00,919611.5,-97.863542,53.696359,"POLYGON ((-98.80636 53.88021, -98.80578 53.880..."
4,4,5,-92.214035,-84.354030,46.410309,49.018391,Superior,United States of America,North America,SWBD,...,1,146.7,2869.953,48410.3,179,-1.00,209219.5,-84.460547,46.468593,"POLYGON ((-90.72250 46.65740, -90.72458 46.657..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427683,1427683,1427684,169.142369,169.151056,-52.593056,-52.585232,None,New Zealand,Oceania,SWBD,...,3,5.1,0.125,139.6,1,4.83,4.4,169.147152,-52.592307,"POLYGON ((169.14763 -52.59291, 169.14750 -52.5..."
1427684,1427684,1427685,73.300817,73.306049,-52.974682,-52.969352,None,Australia,Oceania,SWBD,...,3,2.1,-9999.000,-9999.0,1,1.04,-9999.0,73.305108,-52.973622,"POLYGON ((73.30605 -52.97318, 73.30540 -52.974..."
1427685,1427685,1427686,158.884943,158.892983,-54.531169,-54.525520,None,Australia,Oceania,SWBD,...,3,3.7,0.013,792.4,205,2.63,0.2,158.892278,-54.529970,"POLYGON ((158.89124 -54.53117, 158.88696 -54.5..."
1427686,1427686,1427687,158.883212,158.897343,-54.597979,-54.588744,None,Australia,Oceania,SWBD,...,3,6.5,0.020,1263.9,163,7.58,0.4,158.889583,-54.594300,"POLYGON ((158.88858 -54.59767, 158.88833 -54.5..."


In [ ]:
ALIASES = {
    "alqueva": "alqueva reservoir",
    "bourget": "lac du bourget",
    "dongqian": "dongqian lake",
    "doninos": "laguna de doninos",
    "iseo": "lago d iseo",
    "kasumigaura": "lake kasumigaura",
    "lugano": "lake lugano",
    "lunz": "lunzer see",
    "maggiore": "lago maggiore",
    "meirama": "lago de meirama",
    "mogan": "mogan golu",
    "mozhaysz": "mozhayskoye reservoir",
    "neagh": "lough neagh",
    "peipsi": "lake peipus",
    "plastic": "lake plastiras",
    "rivadavia": "lago rivadavia",
    "rotorua": "lake rotorua",
    "sau": "sau reservoir",
    "sibaya": "lake sibaya",
    "stechlin": "stechlinsee",
    "taharoa": "lake taharoa",
    "tahoe": "lake tahoe",
    "tanganyika": "lake tanganyika",
    "vortstjarv": "vortsjarv",
    "wdzydze": "jezioro wdzydze",
    "windermere": "windermere",
    "wiritoa": "lake wiritoa",
}
LAKE_LIST = [
"Alqueva","Avery","Azul","Bourget","Bueng Boraphet","Cecebre","Clinton","Dickie",
"Dongqian","Doninos","Erken","Esrum","Forest","Iseo","Kasumigaura","Lugano","Lunz",
"Maggiore","McNearny","Meirama","Mogan","Mozhaysz","Neagh","North","Peipsi","Plastic",
"Rivadavia","Rotorua","Sau","Sibaya","Stechlin","Taharoa","Tahoe","Tanganyika",
"Vortstjarv","Wdzydze","Windermere","Wiritoa"
]

def norm(s):
    s = unidecode(str(s)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s
targets = []
for n in LAKE_LIST:
    k = norm(n)
    targets.append(ALIASES.get(k, k))
    targets.append(k)
targets = pd.unique(targets).tolist()
# targets+=LAKE_LIST

targets

In [ ]:
df3['lake_lower']=df3['Lake_name'].apply(norm)


df3[df3['Hylak_id'].isin(lake_id)].to_excel('target_lake_info.xlsx')